# Chess Assistant — train / fine-tune model nhận diện quân trên Google Colab

Chạy trên GPU của Google, máy bạn không phải chạy nặng.

**Chuẩn bị (trên máy bạn):**
1. Gộp dữ liệu: `python -m scripts.prepare_dataset --sources "data/public/*" "data/captured/*" --out data/pieces_v3`
2. Đóng gói: `python -m scripts.export_colab --dataset data/pieces_v3` → tạo thư mục `colab_upload/`
3. Tải **các file trong** `colab_upload/` (`pieces_dataset.zip`, `pieces.pt`) lên Google Drive vào thư mục **`MyDrive/ChessAssistant/`**

**Trên Colab:** `Runtime → Change runtime type → T4 GPU`, rồi chạy lần lượt từng ô (`Shift+Enter`).
Kết quả lưu về Drive: `MyDrive/ChessAssistant/pieces_new.pt` → tải về, đặt vào `models/pieces.pt` trên máy.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cài Ultralytics và kết nối Google Drive

In [ ]:
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## 3. Cấu hình
- `BASE_MODEL`: mặc định fine-tune tiếp từ `pieces.pt` trên Drive (nhanh, giữ những gì model đã học). Không có thì train từ `yolo11s.pt`.
- `EPOCHS`: fine-tune 30 là đủ; train từ đầu nên 100.
- `RESUME = True` nếu lần trước Colab bị ngắt giữa chừng — sẽ train tiếp từ checkpoint đã lưu trên Drive.

In [ ]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/ChessAssistant')
DATASET_ZIP = DRIVE_DIR / 'pieces_dataset.zip'
BASE_MODEL = DRIVE_DIR / 'pieces.pt'
RUNS_DIR = DRIVE_DIR / 'runs'      # lưu trên Drive để không mất khi Colab ngắt
RUN_NAME = 'finetune'

EPOCHS = 30
BATCH = 32        # T4 16GB chạy được 32 với ảnh 640; báo thiếu bộ nhớ thì hạ xuống 16
IMGSZ = 640
RESUME = False

assert DATASET_ZIP.exists(), f'Không thấy {DATASET_ZIP} — tải pieces_dataset.zip lên Drive trước'
print('Dataset:', DATASET_ZIP, f'({DATASET_ZIP.stat().st_size / 1e6:.0f} MB)')
print('Model gốc:', BASE_MODEL if BASE_MODEL.exists() else 'yolo11s.pt (train từ đầu)')

## 4. Giải nén dataset vào ổ cục bộ của Colab (đọc nhanh hơn Drive)

In [ ]:
import shutil, zipfile, yaml

DATA_DIR = Path('/content/pieces')
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(DATA_DIR)

data_yaml = DATA_DIR / 'data.yaml'
cfg = yaml.safe_load(data_yaml.read_text())
cfg['path'] = str(DATA_DIR)
data_yaml.write_text(yaml.safe_dump(cfg, allow_unicode=True))

for split in ('train', 'val', 'test'):
    n = len(list((DATA_DIR / split / 'images').glob('*'))) if (DATA_DIR / split / 'images').exists() else 0
    print(f'{split}: {n} ảnh')
print('Lớp:', cfg['names'])

## 5. Train / fine-tune

In [ ]:
from ultralytics import YOLO

last = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
if RESUME and last.exists():
    model = YOLO(str(last))
    results = model.train(resume=True)
else:
    model = YOLO(str(BASE_MODEL) if BASE_MODEL.exists() else 'yolo11s.pt')
    results = model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        close_mosaic=10,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        patience=15,
    )
best = Path(results.save_dir) / 'weights' / 'best.pt'
print('Model tốt nhất:', best)

## 6. Đánh giá trên tập test
`mAP50` gần 1 là tốt. **Lưu ý:** con số này chỉ phản ánh loại ảnh có trong tập test. Nếu ảnh game 3D chỉ nằm trong tập train
(vd. dùng `--holdout` cho một nguồn khác), chỉ số ở đây không nói gì về game 3D.

In [ ]:
best_model = YOLO(str(best))
metrics = best_model.val(data=str(data_yaml), split='test', imgsz=IMGSZ, batch=BATCH, verbose=False)
print(f'mAP50 = {metrics.box.map50:.3f} | mAP50-95 = {metrics.box.map:.3f} | P = {metrics.box.mp:.3f} | R = {metrics.box.mr:.3f}')
for i, ap50 in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f'  {best_model.names[int(i)]}: {ap50:.3f}')

## 6b. Đánh giá riêng từng nguồn dữ liệu
Chỉ số chung ở trên có thể rất cao chỉ vì nguồn chiếm đa số (vd. ChessReD) dễ — trong khi nguồn khác (cờ 2D, game 3D) kém.
Tên ảnh có tiền tố nguồn (`chessred2k_`, `chess2d_`, `game3d_`...) nên tách ra đánh giá từng nhóm.

In [ ]:
from collections import defaultdict

groups = defaultdict(list)
for img in sorted((DATA_DIR / 'test' / 'images').glob('*')):
    groups[img.name.split('_')[0]].append(str(img))

for source, imgs in groups.items():
    list_file = DATA_DIR / f'test_{source}.txt'
    list_file.write_text('\n'.join(imgs))
    sub_yaml = DATA_DIR / f'data_{source}.yaml'
    sub_yaml.write_text(yaml.safe_dump({**cfg, 'test': str(list_file)}, allow_unicode=True))
    m = best_model.val(data=str(sub_yaml), split='test', imgsz=IMGSZ, batch=BATCH, verbose=False, plots=False)
    print(f'{source:12s} {len(imgs):5d} ảnh | mAP50 {m.box.map50:.3f} | P {m.box.mp:.3f} | R {m.box.mr:.3f}')

## 7. Xem thử vài dự đoán trên ảnh test

In [ ]:
import random
from IPython.display import Image, display

test_imgs = sorted((DATA_DIR / 'test' / 'images').glob('*'))
for p in random.sample(test_imgs, min(3, len(test_imgs))):
    r = best_model.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    out = f'/content/pred_{p.stem}.jpg'
    r.save(filename=out)
    print(p.name, '-', len(r.boxes), 'quân')
    display(Image(out, width=700))

## 8. Lưu model về Drive

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
dst = DRIVE_DIR / 'pieces_new.pt'
shutil.copy2(best, dst)
shutil.copy2(best, DRIVE_DIR / f'pieces_{stamp}.pt')   # bản lưu theo thời gian
print('Đã lưu:', dst)
print('Tải pieces_new.pt về máy, đổi tên thành pieces.pt, đặt vào thư mục models/ của dự án '
      '(nhớ giữ bản cũ: models/pieces_v1.pt).')